In [ ]:
import time, json
import numpy as np
import open_clip, torch
from qdrant_client import QdrantClient
from deep_translator import GoogleTranslator
from sentence_transformers import SentenceTransformer
from IPython.display import display, Image as IPImage

# ══════════════════════════════════════════════════════
# 1. Qdrant 연결
# ══════════════════════════════════════════════════════
qdrant_visual = QdrantClient(path=r"로컬 경로로 변경/qdrant_storage")
print("visual:", qdrant_visual.get_collection("visual").points_count)

# ══════════════════════════════════════════════════════
# 2. 캡션 로드
# ══════════════════════════════════════════════════════
captions = {}
captions_cat = {}
with open(r"로컬경로로 변경/captions_full_lite.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        fid = str(data['file_id'])
        captions[fid] = (
            data['caption_category'] + ' ' +
            ' '.join(data.get('caption_micro_details', [])) + ' ' +
            ' '.join(data.get('mood_and_tpo', []))
        ).strip()
        captions_cat[fid] = data['category']
print(f"캡션 수: {len(captions)}")

# ══════════════════════════════════════════════════════
# 3. 모델 로드
# ══════════════════════════════════════════════════════
model, _, _ = open_clip.create_model_and_transforms('hf-hub:Marqo/marqo-fashionSigLIP')
clip_tokenizer = open_clip.get_tokenizer('hf-hub:Marqo/marqo-fashionSigLIP')
reranker = SentenceTransformer("jhgan/ko-sroberta-multitask")
print("모델 로드 완료!")

# ══════════════════════════════════════════════════════
# 4. 사전들
# ══════════════════════════════════════════════════════
fashion_dict = {
    "비침 없는": "opaque", "비침없는": "opaque",
    "비침 있는": "sheer", "비침있는": "sheer",
    "올풀림": "frayed", "오버핏": "oversized",
    "잔잔한": "small subtle", "잔꽃": "ditsy floral",
}

attribute_groups = {
    "pattern":  ["스트라이프", "체크", "도트", "플로럴", "지브라", "레오파드"],
    "texture":  ["주름", "플리츠", "셔링", "골지", "퀼팅", "러플"],
    "color":    ["화이트", "블랙", "베이지", "네이비", "그레이", "브라운", "카키", "아이보리", "라벤더", "민트"],
    "material": ["맨투맨", "스웨트", "데님", "쉬폰", "린넨", "울", "코튼", "가디건", "블라우스", "후드", "니트"],
    "finish":   ["시스루", "컷오프", "프린지", "데미지"],
    "length":   ["미디", "롱"],
}

negation_map = {
    "비침 없는": ("finish", ["시스루"]), "비침없는": ("finish", ["시스루"]),
}
positive_map = {
    "비침 있는": ("finish", ["시스루"]), "비침있는": ("finish", ["시스루"]),
    "비치는": ("finish", ["시스루"]),
    "올풀림": ("finish", ["컷오프", "프린지", "데미지"]),
}
conflict_pairs = [("pattern", "texture")]

category_keywords = {
    "top":       ["셔츠", "블라우스", "티셔츠", "니트", "맨투맨", "후드", "탑", "탱크탑"],
    "bottom":    ["팬츠", "바지", "스커트", "치마", "데님", "청바지", "슬랙스"],
    "dress":     ["원피스", "드레스"],
    "outerwear": ["코트", "자켓", "재킷", "패딩", "점퍼", "가디건", "블레이저"],
}

# ══════════════════════════════════════════════════════
# 5. 함수 정의
# ══════════════════════════════════════════════════════
def translate_to_english(query):
    q = query
    for ko, en in fashion_dict.items():
        q = q.replace(ko, en)
    translated = GoogleTranslator(source='ko', target='en').translate(q)
    fashion_query = f"fashion product photo of {translated}"
    print(f"번역: {query} → {fashion_query}")
    return fashion_query

def encode_text_clip(query):
    query_en = translate_to_english(query)
    text_input = clip_tokenizer([query_en])
    with torch.no_grad():
        emb = model.encode_text(text_input)
    emb = emb.squeeze()
    emb = emb / emb.norm()
    return emb.numpy()

def extract_category(query):
    for cat, keywords in category_keywords.items():
        for kw in keywords:
            if kw in query:
                return cat
    return None

def extract_query_attrs(query):
    query_attrs, must_not, must_have = {}, {}, {}
    for phrase, (group, kws) in negation_map.items():
        if phrase in query:
            must_not.setdefault(group, []).extend(kws)
    for phrase, (group, kws) in positive_map.items():
        if phrase in query:
            must_have.setdefault(group, []).extend(kws)
    for group, keywords in attribute_groups.items():
        for kw in keywords:
            if kw in query:
                query_attrs.setdefault(group, []).append(kw)
    return query_attrs, must_not, must_have

def _caption_filter_logic(caption, query_attrs, must_not, must_have, penalty, boost_val, file_cat=None):
    boost = 1.0
    for group, kws in must_not.items():
        if any(k in caption for k in kws):
            boost *= penalty
    for group, kws in must_have.items():
        boost *= boost_val if any(k in caption for k in kws) else penalty
    for group, kws in query_attrs.items():
        if group == "length" and file_cat not in ["bottom", "dress"]:
            continue
        caption_has = [k for k in attribute_groups[group] if k in caption]
        if caption_has:
            boost *= boost_val if any(k in caption for k in kws) else penalty
    for g1, g2 in conflict_pairs:
        if g1 in query_attrs:
            has_g1 = any(k in caption for k in attribute_groups[g1])
            has_g2 = any(k in caption for k in attribute_groups[g2])
            if has_g2 and not has_g1:
                boost *= penalty
    return boost

def apply_caption_filter(query, candidates, penalty=0.5, boost_val=1.3):
    query_attrs, must_not, must_have = extract_query_attrs(query)
    scored = []
    for id_, base_score in candidates:
        payload = qdrant_visual.retrieve(
            collection_name="visual", ids=[id_], with_payload=True
        )[0].payload
        filename = payload['filename']
        file_id = filename.split('_')[0]
        file_cat = filename.split('_')[1].split('.')[0]
        cap_cat = captions_cat.get(file_id)
        caption = captions.get(file_id, "") if (cap_cat == file_cat) else ""
        boost = _caption_filter_logic(
            caption, query_attrs, must_not, must_have, penalty, boost_val, file_cat=file_cat
        ) if caption else 1.0
        scored.append((id_, base_score * boost))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored

def rerank_with_kosroberta(query, candidates, top_n=10):
    """ko-sroberta로 쿼리-캡션 유사도 계산해서 재정렬"""
    query_emb = reranker.encode(query, normalize_embeddings=True)
    scored = []
    for id_, base_score in candidates:
        payload = qdrant_visual.retrieve(
            collection_name="visual", ids=[id_], with_payload=True
        )[0].payload
        filename = payload['filename']
        file_id = filename.split('_')[0]
        file_cat = filename.split('_')[1].split('.')[0]
        cap_cat = captions_cat.get(file_id)
        caption = captions.get(file_id, "") if (cap_cat == file_cat) else ""
        if caption:
            cap_emb = reranker.encode(caption, normalize_embeddings=True)
            sim = float(np.dot(query_emb, cap_emb))
            combined = base_score * 0.5 + sim * 0.5
        else:
            combined = base_score
        scored.append((id_, combined))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_n]

def _get_visual_candidates(query, top_k=100, use_category_filter=False):
    v_emb = encode_text_clip(query)
    hits = qdrant_visual.query_points(
        collection_name="visual", query=v_emb.tolist(), limit=top_k
    ).points
    if use_category_filter:
        target_cat = extract_category(query)
        if target_cat:
            filtered = []
            for h in hits:
                payload = qdrant_visual.retrieve(
                    collection_name="visual", ids=[h.id], with_payload=True
                )[0].payload
                if target_cat in payload.get('filename', ''):
                    filtered.append(h)
            hits = filtered
    return [(h.id, h.score) for h in hits], v_emb

def display_results(results, top_n=10):
    for rank, (id_, score) in enumerate(results[:top_n], 1):
        payload = qdrant_visual.retrieve(
            collection_name="visual", ids=[id_], with_payload=True
        )[0].payload
        filename = payload['filename']
        file_id = filename.split('_')[0]
        caption = captions.get(file_id, "(캡션 없음)")
        url = f"https://pub-5966bf5d84f948c983500b6d9547eec9.r2.dev/masking_data/{filename}"
        print(f"#{rank} | {filename} | 점수: {score:.4f}")
        print(f"캡션: {caption}")
        display(IPImage(url=url, width=200))
        print()

print("함수 정의 완료!")

# ══════════════════════════════════════════════════════
# 6. 최종 검색 함수
# ══════════════════════════════════════════════════════
def search(query, top_k=100, top_n=10):
    """최종 모델: visual → 카테고리 필터 → 캡션 필터링 → ko-sroberta 리랭킹"""
    start = time.time()
    candidates, _ = _get_visual_candidates(query, top_k=top_k, use_category_filter=True)
    print(f"카테고리 필터 후: {len(candidates)}개")
    filtered = apply_caption_filter(query, candidates)[:50]
    results = rerank_with_kosroberta(query, filtered, top_n=top_n)
    print(f"검색 시간: {int((time.time()-start)*1000)}ms")
    return results



In [ ]:
# ══════════════════════════════════════════════════════
# 7. 검색 실행
# ══════════════════════════════════════════════════════
query = "흰 스티치 생지 데님 스트레이트"
results = search(query)
display_results(results)